In [ ]:
# This notebook requires the Schemdraw package to be installed.

%matplotlib widget

import numpy as np
import sympy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import schemdraw
import schemdraw.elements as elm
from matplotlib.ticker import NullFormatter
from ipywidgets import FloatSlider, RadioButtons, HBox, VBox, Layout, HTML, HTMLMath
from IPython.display import display

mpl.rcParams['figure.max_open_warning'] = 50

s, R, L, C = sp.symbols('s R L C', positive=True, real=True)

# ------------------------------------------------------------
# TRANSFER FUNCTIONS
# ------------------------------------------------------------

def get_transfer_function(filter_type):
    if filter_type == 'Low-pass':
        H = 1 / (L*C*s**2 + (L/R)*s + 1)
        wc = 1 / sp.sqrt(L*C)
        Q = R * sp.sqrt(C/L)

    elif filter_type == 'High-pass':
        H = L*C*s**2 / (L*C*s**2 + (L/R)*s + 1)
        wc = 1 / sp.sqrt(L*C)
        Q = R * sp.sqrt(C/L)

    elif filter_type == 'Band-pass':
        H = L*s / (R*L*C*s**2 + L*s + R)
        wc = 1 / sp.sqrt(L*C)
        Q = R * sp.sqrt(C/L)

    elif filter_type == 'Band-stop':
        H = (L*C*s**2 + 1) / (L*C*s**2 + R*C*s + 1)
        wc = 1 / sp.sqrt(L*C)
        Q = sp.sqrt(L/C) / R

    return sp.factor(H), sp.simplify(wc), sp.simplify(Q)

# ------------------------------------------------------------
# CONTROLS
# ------------------------------------------------------------

filter_title = HTML(value="<b>Filter Type:</b>")
filter_radio = RadioButtons(options=['Low-pass', 'High-pass', 'Band-pass', 'Band-stop'], value='Low-pass', description='', layout=Layout(width='150px'))

r_title = HTML(value="<b style='color:red;'>Resistance R (Ω)</b>")
r_slider = FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='', readout=True, readout_format='.2f', continuous_update=True, style={'handle_color': 'red'}, layout=Layout(width='240px'))

l_title = HTML(value="<b style='color:blue;'>Inductance L (H)</b>")
l_slider = FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='', readout=True, readout_format='.2f', continuous_update=True, style={'handle_color': 'blue'}, layout=Layout(width='240px'))

c_title = HTML(value="<b style='color:green;'>Capacitance C (F)</b>")
c_slider = FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='', readout=True, readout_format='.2f', continuous_update=True, style={'handle_color': 'green'}, layout=Layout(width='240px'))

# ------------------------------------------------------------
# EQUATIONS
# ------------------------------------------------------------

equation_title = HTML(value="<b>Transfer Function and Parameters</b>")
equation_display = HTMLMath(layout=Layout(width='300px'))

def update_equations():
    filter_type = filter_radio.value
    R_val = r_slider.value
    L_val = l_slider.value
    C_val = c_slider.value

    H_sym, wc_sym, Q_sym = get_transfer_function(filter_type)

    wc_val = float(wc_sym.subs({R: R_val, L: L_val, C: C_val}))
    Q_val = float(Q_sym.subs({R: R_val, L: L_val, C: C_val}))

    equation_display.value = r"$$\mathcal{H}(s)=" + sp.latex(H_sym) + r"$$" + r"$$\omega_c=" + sp.latex(wc_sym) + rf"={wc_val:.4f}\ \mathrm{{rad/s}}$$" + r"$$Q=" + sp.latex(Q_sym) + rf"={Q_val:.4f}$$"

# ------------------------------------------------------------
# CIRCUIT
# ------------------------------------------------------------

circuit_title = HTML(value="<b>Circuit</b>")
circuit_display = HTML(layout=Layout(width='380px', height='170px', overflow='hidden'))

def make_circuit_svg(filter_type):
    d = schemdraw.Drawing(show=False, unit=1.6)

    if filter_type == 'Low-pass':
        d += elm.Dot().at((0, 0))
        d += elm.Line().right().length(0.4)
        d += elm.Inductor().right().length(1.6).label('$L$', loc='top').color('blue')
        d += elm.Line().right().length(1.0)
        d += elm.Dot()
        node = d.here

        d += elm.Line().right().length(0.8)
        d += elm.Dot().label('$V_{out}$', loc='right')

        d += elm.Capacitor().at(node).down().length(2.0).label('$C$', loc='left').color('green')
        bottom_c = d.here

        d += elm.Resistor().at((node[0] + 0.8, node[1])).down().length(2.0).label('$R$', loc='right').color('red')
        bottom_r = d.here

        d += elm.Line().at(bottom_c).right().to(bottom_r)
        d += elm.Line().at((0, -2.0)).right().to(bottom_r)
        d += elm.Dot().at((0, -2.0))
        d += elm.Dot().at(bottom_r)
        d += elm.Label().at((-0.15, -1.0)).label('$V_{in}$')

    elif filter_type == 'High-pass':
        d += elm.Dot().at((0, 0))
        d += elm.Line().right().length(0.4)
        d += elm.Capacitor().right().length(1.6).label('$C$', loc='top').color('green')
        d += elm.Line().right().length(1.0)
        d += elm.Dot()
        node = d.here

        d += elm.Line().right().length(0.8)
        d += elm.Dot().label('$V_{out}$', loc='right')

        d += elm.Inductor().at(node).down().length(2.0).label('$L$', loc='left').color('blue')
        bottom_l = d.here

        d += elm.Resistor().at((node[0] + 0.8, node[1])).down().length(2.0).label('$R$', loc='right').color('red')
        bottom_r = d.here

        d += elm.Line().at(bottom_l).right().to(bottom_r)
        d += elm.Line().at((0, -2.0)).right().to(bottom_r)
        d += elm.Dot().at((0, -2.0))
        d += elm.Dot().at(bottom_r)
        d += elm.Label().at((-0.15, -1.0)).label('$V_{in}$')

    elif filter_type == 'Band-pass':
        d += elm.Dot().at((0, 0))
        d += elm.Line().right().length(0.4)
        d += elm.Resistor().right().length(1.6).label('$R$', loc='top').color('red')
        d += elm.Line().right().length(1.0)
        d += elm.Dot()
        node = d.here

        d += elm.Line().right().length(0.8)
        d += elm.Dot().label('$V_{out}$', loc='right')

        d += elm.Inductor().at(node).down().length(2.0).label('$L$', loc='left').color('blue')
        bottom_l = d.here

        d += elm.Capacitor().at((node[0] + 0.8, node[1])).down().length(2.0).label('$C$', loc='right').color('green')
        bottom_c = d.here

        d += elm.Line().at(bottom_l).right().to(bottom_c)
        d += elm.Line().at((0, -2.0)).right().to(bottom_c)
        d += elm.Dot().at((0, -2.0))
        d += elm.Dot().at(bottom_c)
        d += elm.Label().at((-0.15, -1.0)).label('$V_{in}$')

    elif filter_type == 'Band-stop':
        d += elm.Dot().at((0, 0))
        d += elm.Line().right().length(0.4)
        d += elm.Resistor().right().length(1.6).label('$R$', loc='top').color('red')
        d += elm.Line().right().length(1.0)
        d += elm.Dot()
        node = d.here

        d += elm.Line().right().length(0.8)
        d += elm.Dot().label('$V_{out}$', loc='right')

        d += elm.Inductor().at(node).down().length(0.95).label('$L$', loc='left').color('blue')
        d += elm.Capacitor().down().length(1.05).label('$C$', loc='left').color('green')
        bottom_lc = d.here

        d += elm.Line().at((0, -2.0)).right().length(node[0] + 0.8)
        d += elm.Dot().at((0, -2.0))
        d += elm.Dot().at((node[0] + 0.8, -2.0))
        d += elm.Line().at(bottom_lc).right().to((node[0] + 0.8, -2.0))
        d += elm.Label().at((-0.15, -1.0)).label('$V_{in}$')

    return d.get_imagedata('svg').decode('utf-8')

def update_circuit():
    circuit_display.value = "<div style='width:100%;height:165px;overflow:hidden;'>" + make_circuit_svg(filter_radio.value) + "</div>"

# ------------------------------------------------------------
# TOP ROW
# ------------------------------------------------------------

filter_box = VBox([filter_title, filter_radio], layout=Layout(width='150px'))
equation_box = VBox([equation_title, equation_display], layout=Layout(width='310px'))
circuit_box = VBox([circuit_title, circuit_display], layout=Layout(width='390px'))

top_row = HBox([filter_box, equation_box, circuit_box], layout=Layout(width='880px', align_items='flex-start', justify_content='space-between'))

# ------------------------------------------------------------
# SLIDERS
# ------------------------------------------------------------

r_box = VBox([r_title, r_slider], layout=Layout(width='250px'))
l_box = VBox([l_title, l_slider], layout=Layout(width='250px'))
c_box = VBox([c_title, c_slider], layout=Layout(width='250px'))

slider_row = HBox([r_box, l_box, c_box], layout=Layout(width='800px', justify_content='space-between', margin='8px 0 12px 0'))

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.ioff()

fig_mag, ax_mag = plt.subplots(figsize=(3.4, 2.4))
fig_phase, ax_phase = plt.subplots(figsize=(3.4, 2.4))
fig_group, ax_group = plt.subplots(figsize=(3.4, 2.4))

line_mag, = ax_mag.semilogx([], [], 'r-', linewidth=1.7)
line_phase, = ax_phase.semilogx([], [], 'r-', linewidth=1.7)
line_group, = ax_group.semilogx([], [], 'r-', linewidth=1.7)

wc_line_mag = ax_mag.axvline(1.0, color='gray', linestyle='--', linewidth=1)
wc_line_phase = ax_phase.axvline(1.0, color='gray', linestyle='--', linewidth=1)
wc_line_group = ax_group.axvline(1.0, color='gray', linestyle='--', linewidth=1)

ax_mag.set_title('Magnitude Response')
ax_mag.set_xlabel('Frequency ω (rad/s)')
ax_mag.set_ylabel('Gain (dB)')
ax_mag.grid(True, which='both', linestyle=':', alpha=0.7)

ax_phase.set_title('Phase Response')
ax_phase.set_xlabel('Frequency ω (rad/s)')
ax_phase.set_ylabel('Phase (deg)')
ax_phase.grid(True, which='both', linestyle=':', alpha=0.7)

ax_group.set_title('Group Delay')
ax_group.set_xlabel('Frequency ω (rad/s)')
ax_group.set_ylabel('Group Delay (s)')
ax_group.grid(True, which='both', linestyle=':', alpha=0.7)

for ax in [ax_mag, ax_phase, ax_group]:
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.set_frame_on(True)

    for spine in ['left', 'right', 'top', 'bottom']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(0.8)
        ax.spines[spine].set_clip_on(False)

for fig in [fig_mag, fig_phase, fig_group]:
    fig.subplots_adjust(left=0.20, right=0.94, bottom=0.24, top=0.86)

for canvas in [fig_mag.canvas, fig_phase.canvas, fig_group.canvas]:
    canvas.toolbar_visible = False
    canvas.header_visible = False
    canvas.footer_visible = False
    canvas.resizable = False
    canvas.layout.width = '340px'
    canvas.layout.height = '240px'
    canvas.layout.overflow = 'visible'

# ------------------------------------------------------------
# PLOT LAYOUT
# ------------------------------------------------------------

mag_box = VBox([fig_mag.canvas], layout=Layout(width='340px', overflow='visible'))
phase_box = VBox([fig_phase.canvas], layout=Layout(width='340px', overflow='visible'))
group_box = VBox([fig_group.canvas], layout=Layout(width='340px', overflow='visible'))

plot_row = HBox([mag_box, phase_box, group_box], layout=Layout(width='1040px', justify_content='space-between', align_items='center', overflow='visible'))

# ------------------------------------------------------------
# INFORMATION MESSAGE
# ------------------------------------------------------------

info_message = HTML(value='', layout=Layout(width='1040px', margin='4px 0 0 0'))

def update_info_message():
    if filter_radio.value == 'Band-stop':
        info_message.value = """
        <div style="font-size:14px; line-height:1.5; padding:8px 12px;">
        <b>Note on the group delay:</b>
        At the notch frequency, the magnitude response becomes zero, so the phase is undefined at that frequency.
        Since group delay is defined as the negative derivative of phase with respect to angular frequency,
        a direct numerical differentiation produces an artificial very large spike near the notch.
        To avoid displaying this numerical artifact, group-delay values are omitted in a very small neighborhood
        where the magnitude response is effectively zero.
        </div>
        """
    else:
        info_message.value = ''

# ------------------------------------------------------------
# UPDATE PLOTS
# ------------------------------------------------------------

def update_plots():
    filter_type = filter_radio.value
    R_val = r_slider.value
    L_val = l_slider.value
    C_val = c_slider.value

    H_sym, wc_sym, Q_sym = get_transfer_function(filter_type)
    wc_val = float(wc_sym.subs({R: R_val, L: L_val, C: C_val}))

    omega = np.logspace(np.log10(wc_val / 10.0), np.log10(wc_val * 10.0), 2000)
    jw = 1j * omega

    if filter_type == 'Low-pass':
        H = 1.0 / (L_val*C_val*jw**2 + (L_val/R_val)*jw + 1.0)

    elif filter_type == 'High-pass':
        H = L_val*C_val*jw**2 / (L_val*C_val*jw**2 + (L_val/R_val)*jw + 1.0)

    elif filter_type == 'Band-pass':
        H = L_val*jw / (R_val*L_val*C_val*jw**2 + L_val*jw + R_val)

    elif filter_type == 'Band-stop':
        H = (L_val*C_val*jw**2 + 1.0) / (L_val*C_val*jw**2 + R_val*C_val*jw + 1.0)

    magnitude_db = 20.0 * np.log10(np.maximum(np.abs(H), 1e-12))
    phase = np.unwrap(np.angle(H))
    phase_deg = np.degrees(phase)
    group_delay = -np.gradient(phase, omega)

    if filter_type == 'Band-stop':
        group_delay[np.abs(H) < 1e-3] = np.nan

    line_mag.set_data(omega, magnitude_db)
    line_phase.set_data(omega, phase_deg)
    line_group.set_data(omega, group_delay)

    wc_line_mag.set_xdata([wc_val, wc_val])
    wc_line_phase.set_xdata([wc_val, wc_val])
    wc_line_group.set_xdata([wc_val, wc_val])

    xmin = wc_val / 10.0
    xmax = wc_val * 10.0

    ax_mag.set_xlim(xmin, xmax)
    ax_phase.set_xlim(xmin, xmax)
    ax_group.set_xlim(xmin, xmax)

    major_ticks = wc_val * np.array([0.1, 1.0, 10.0])
    minor_ticks = wc_val * np.concatenate((np.arange(2, 10) / 10.0, np.arange(2, 10)))

    for ax in [ax_mag, ax_phase, ax_group]:
        ax.set_xticks(major_ticks)
        ax.set_xticks(minor_ticks, minor=True)
        ax.xaxis.set_minor_formatter(NullFormatter())

    ax_mag.relim()
    ax_mag.autoscale_view(scalex=False, scaley=True)

    ax_phase.relim()
    ax_phase.autoscale_view(scalex=False, scaley=True)

    ax_group.relim()
    ax_group.autoscale_view(scalex=False, scaley=True)

    fig_mag.canvas.draw_idle()
    fig_phase.canvas.draw_idle()
    fig_group.canvas.draw_idle()

# ------------------------------------------------------------
# CALLBACKS
# ------------------------------------------------------------

def slider_changed(change):
    update_equations()
    update_plots()

def filter_changed(change):
    update_equations()
    update_circuit()
    update_plots()
    update_info_message()

r_slider.observe(slider_changed, names='value')
l_slider.observe(slider_changed, names='value')
c_slider.observe(slider_changed, names='value')
filter_radio.observe(filter_changed, names='value')

# ------------------------------------------------------------
# INITIALIZATION
# ------------------------------------------------------------

update_equations()
update_circuit()
update_plots()
update_info_message()

display(HTML("""
<style>
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
}
.jp-OutputArea-child {
    overflow: visible !important;
    max-height: none !important;
}
.jp-OutputArea {
    overflow: visible !important;
    max-height: none !important;
}
</style>
"""))

display(top_row)
display(slider_row)
display(plot_row)
display(info_message)

for fig in [fig_mag, fig_phase, fig_group]:
    fig.canvas.draw()